# Quantum Computing Module
## Advanced Quantum Algorithms and Simulations

This notebook covers:

1. Grover's Algorithm (Amplitude Amplification)
2. Quantum Fourier Transform (QFT)
3. Quantum Phase Estimation
4. Shor's Algorithm (Simplified)
5. Variational Quantum Eigensolver (VQE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
from IPython.display import display, Math

print("Quantum Computing Module Loaded")
print("=" * 50)

## 1. Grover's Algorithm - Detailed Analysis

Grover's search algorithm provides quadratic speedup for unstructured database search.

In [ ]:
def grovers_iteration(state, oracle):
    """Single Grover iteration: Oracle + Diffusion"""
    # Apply oracle
    after_oracle = oracle @ state
    
    # Diffusion operator
    n = len(state)
    diffusion = 2 * np.ones((n, n)) / n - np.eye(n)
    after_diffusion = diffusion @ after_oracle
    
    return after_diffusion

def grovers_search(n_qubits, target_states, max_iterations=100):
    """
    Grover's algorithm for multiple targets.
    """
    N = 2 ** n_qubits
    
    # Create multi-target oracle
    oracle = np.eye(N)
    for t in target_states:
        oracle[t, t] = -1
    
    # Initial superposition state
    H_n = np.kron(np.array([[1, 1], [1, -1]]) / np.sqrt(2), 
                 np.array([[1, 1], [1, -1]]) / np.sqrt(2))
    for _ in range(n_qubits - 2):
        H_n = np.kron(H_n, np.array([[1, 1], [1, -1]]) / np.sqrt(2))
    
    state = H_n @ np.eye(N)[:, 0]
    
    # Optimal iterations
    M = len(target_states)
    optimal_iters = int(np.round(np.pi / 4 * np.sqrt(N / M)))
    optimal_iters = min(optimal_iters, max_iterations)
    
    # Store probability evolution
    probs_history = [np.abs(state) ** 2]
    
    for i in range(optimal_iters):
        state = grovers_iteration(state, oracle)
        probs_history.append(np.abs(state) ** 2)
    
    # Measure
    probs = np.abs(state) ** 2
    result = np.random.choice(N, p=probs / probs.sum())
    
    return result, probs_history, optimal_iters

# Example with 8 qubits, 2 target states
n_qubits = 8
targets = [42, 137]

result, probs_hist, iters = grovers_search(n_qubits, targets)

print(f"Grover's Search: {n_qubits} qubits, targets: {targets}")
print(f"Optimal iterations: {iters}")
print(f"Result: {result}")
print(f"Target found: {result in targets}")

In [ ]:
# Visualize Grover's probability evolution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Probability evolution over iterations
for t in targets:
    probs_t = [p[t] for p in probs_hist]
    ax1.plot(probs_t, 'o-', label=f'Target {t}', markersize=4)

ax1.set_xlabel('Iteration')
ax1.set_ylabel('Probability')
ax1.set_title("Grover's Amplitude Amplification")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Final probability distribution
final_probs = probs_hist[-1]
ax2.bar(range(0, 256, 1), final_probs, alpha=0.7, color='steelblue')
for t in targets:
    ax2.axvline(t, color='red', linestyle='--', alpha=0.8, linewidth=2)

ax2.set_xlabel('State Index')
ax2.set_ylabel('Probability')
ax2.set_title('Final Probability Distribution')

plt.tight_layout()
plt.show()

## 2. Quantum Fourier Transform (QFT)

The QFT is the quantum analogue of the discrete Fourier transform and is a key component in many quantum algorithms.

In [ ]:
def qft_matrix(n):
    """Create n-qubit QFT matrix"""
    N = 2 ** n
    qft = np.zeros((N, N), dtype=complex)
    
    for i in range(N):
        for j in range(N):
            qft[i, j] = np.exp(2j * np.pi * i * j / N) / np.sqrt(N)
    
    return qft

def qft_circuit(n):
    """Simulate QFT circuit"""
    # Single qubit gates
    H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    
    def R(k):
        return np.array([[1, 0], [0, np.exp(2j * np.pi / 2**k)]])
    
    # Build circuit
    state = np.zeros(2**n, dtype=complex)
    state[0] = 1.0
    
    return state, H, R

# Demonstrate QFT
n = 4
qft = qft_matrix(n)

print(f"QFT Matrix ({n} qubits): {qft.shape}")
print(f"Unitaries: {np.allclose(qft @ qft.conj().T, np.eye(2**n))}")
print(f"Trace normalization: {np.abs(np.trace(qft)):.4f}")

In [ ]:
# Visualize QFT matrix structure
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1, ax2 = axes

# Real and imaginary parts
im1 = ax1.imshow(np.abs(qft), cmap='viridis', aspect='auto')
ax1.set_title('|QFT| Magnitude')
ax1.set_xlabel('Column')
ax1.set_ylabel('Row')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(np.angle(qft), cmap='twilight', aspect='auto')
ax2.set_title('QFT Phase')
ax2.set_xlabel('Column')
ax2.set_ylabel('Row')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## 3. Quantum Phase Estimation

Quantum Phase Estimation (QPE) is used to estimate the eigenvalue of a unitary operator.

In [ ]:
def quantum_phase_estimation(unitary, target_state, n_precision_bits, n_measurement=1000):
    """
    Simulate Quantum Phase Estimation.
    """
    # Get true eigenvalue
    eigenvalues, eigenvectors = np.linalg.eig(unitary)
    
    # Find eigenvector closest to target_state
    overlaps = np.abs(eigenvectors.conj().T @ target_state)
    best_idx = np.argmax(overlaps)
    
    true_phase = np.angle(eigenvalues[best_idx]) / (2 * np.pi)
    
    # Simulate measurement with noise
    estimated_phases = []
    for _ in range(n_measurement):
        # Binary fraction estimation
        phase = 0
        for bit in range(n_precision_bits):
            # Apply controlled-U^(2^bit) and measure
            U_power = linalg.matrix_power(unitary, 2**bit)
            prob_0 = np.abs(1 - np.abs(np.vdot(target_state, U_power @ target_state))**2 * 0.5)
            bit_value = np.random.choice([0, 1], p=[prob_0, 1-prob_0])
            phase += bit_value / 2**(bit + 1)
        estimated_phases.append(phase)
    
    return true_phase, np.mean(estimated_phases), np.std(estimated_phases)

# Example: Phase estimation for a unitary
U = np.array([[1, 0], [0, np.exp(2j * np.pi * 0.375)]])  # Phase = 3/8
target = np.array([0, 1])  # Second eigenstate

true_phase, est_phase, std = quantum_phase_estimation(U, target, n_precision_bits=8)

print(f"True Phase: {true_phase:.4f}")
print(f"Estimated Phase: {est_phase:.4f} ± {std:.4f}")
print(f"Exact Value: 0.375")

## 4. Shor's Algorithm (Simplified)

Shor's algorithm provides exponential speedup for integer factorization on quantum computers.

In [ ]:
def classical_period_finding(a, N):
    """
    Classical period finding for Shor's algorithm simulation.
    r = min{r: a^r mod N = 1}
    """
    if np.gcd(a, N) != 1:
        return np.gcd(a, N)  # Found non-trivial factor
    
    r = 1
    f_val = a % N
    while f_val != 1:
        f_val = (f_val * a) % N
        r += 1
        if r > N:
            return None  # Period too large
    
    return r

def shor_simulation(N):
    """
    Simplified Shor's algorithm simulation.
    """
    if N % 2 == 0:
        return 2, N // 2
    
    # Try random bases
    for _ in range(100):
        a = np.random.randint(2, N - 1)
        
        # Check if a and N are coprime
        if np.gcd(a, N) != 1:
            return np.gcd(a, N), N // np.gcd(a, N)
        
        # Find period
        r = classical_period_finding(a, N)
        
        if r is not None and r % 2 == 0:
            # Compute factor
            candidate = np.gcd(a**(r//2) - 1, N)
            if candidate != 1 and candidate != N:
                return candidate, N // candidate
    
    return None, None

# Factor N = 15
N = 15
f1, f2 = shor_simulation(N)

print(f"Shor's Algorithm Simulation: Factor N = {N}")
print(f"Found factors: {f1}, {f2}")
print(f"Verification: {f1 * f2 == N}")

## 5. Variational Quantum Eigensolver (VQE)

VQE is a hybrid quantum-classical algorithm for finding ground state energies.

In [ ]:
def vqe_simulation(hamiltonian, n_qubits, n_iterations=50):
    """
    Simplified VQE simulation using parameter optimization.
    """
    from scipy.optimize import minimize
    
    def ansatz(params):
        """Simple parameterized circuit"""
        state = np.zeros(2**n_qubits, dtype=complex)
        state[0] = 1.0
        
        # Apply rotations
        for i in range(n_qubits):
            theta = params[i]
            R_y = np.array([[np.cos(theta/2), -np.sin(theta/2)],
                           [np.sin(theta/2), np.cos(theta/2)]])
            if i == 0:
                unitary = R_y
            else:
                unitary = np.kron(unitary, R_y)
        
        return unitary @ state
    
    def cost_function(params):
        state = ansatz(params)
        energy = np.real(np.vdot(state, hamiltonian @ state))
        return energy
    
    # Random initial parameters
    x0 = np.random.uniform(0, 2*np.pi, n_qubits)
    
    # Optimize
    result = minimize(cost_function, x0, method='COBYLA')
    
    return result.fun, result.x

# Example Hamiltonian (H_2-like)
H = np.array([[-1.0, 0], 
              [0, -0.5]])  # Simple 2-level Hamiltonian

ground_energy, optimal_params = vqe_simulation(H, n_qubits=1, n_iterations=100)

print(f"VQE Simulation")
print(f"Computed Ground Energy: {ground_energy:.4f}")
print(f"Exact Ground Energy: {np.min(np.linalg.eigvalsh(H)):.4f}")
print(f"Optimal Parameters: {optimal_params}")

In [ ]:
# Summary of Quantum Algorithms

algorithms_summary = {
    'Grover': {'Speedup': 'O(√N) vs O(N)', 'Qubits': 'Log(N)', 'Use Case': 'Unstructured Search'},
    'QFT': {'Speedup': 'Exponential', 'Qubits': 'N', 'Use Case': 'Phase Estimation'},
    'QPE': {'Speedup': 'Polynomial', 'Qubits': 'Log(ε)', 'Use Case': 'Eigenvalue Problems'},
    'Shor': {'Speedup': 'Sub-exponential', 'Qubits': 'O(N)', 'Use Case': 'Factoring'},
    'VQE': {'Speedup': 'Hybrid', 'Qubits': 'Variable', 'Use Case': 'Chemistry'},
}

print("Quantum Algorithms Summary")
print("=" * 70)
print(f"{'Algorithm':<10} {'Speedup':<25} {'Qubits':<15} {'Use Case':<20}")
print("-" * 70)
for algo, info in algorithms_summary.items():
    print(f"{algo:<10} {info['Speedup']:<25} {info['Qubits']:<15} {info['Use Case']:<20}")